In [1]:
import torch
import torch.nn as nn

batch_size = 2
seq_len = 5
embed_dim = 32
hidden_size = 64

# Инициализация слоя
lstm = nn.LSTM(input_size=embed_dim, hidden_size=hidden_size, batch_first=True)

# Фейковые данные (например, после слоя nn.Embedding)
x = torch.randn(batch_size, seq_len, embed_dim)

# Прямой проход
output, (h_n, c_n) = lstm(x)

print("LSTM output:", output.shape) # [2, 5, 64] - память на каждом шаге слов
print("LSTM h_n:", h_n.shape)       # [1, 2, 64] - финальная краткосрочная память
print("LSTM c_n:", c_n.shape)       # [1, 2, 64] - финальный конвейер долгосрочной памяти

LSTM output: torch.Size([2, 5, 64])
LSTM h_n: torch.Size([1, 2, 64])
LSTM c_n: torch.Size([1, 2, 64])


In [2]:
# Инициализация слоя
gru = nn.GRU(input_size=embed_dim, hidden_size=hidden_size, batch_first=True)

# Прямой проход
output, h_n = gru(x)

print("GRU output:", output.shape)  # [2, 5, 64]
print("GRU h_n:", h_n.shape)        # [1, 2, 64] - единая финальная память

GRU output: torch.Size([2, 5, 64])
GRU h_n: torch.Size([1, 2, 64])


In [3]:
class TextLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # Меняем nn.RNN на nn.LSTM (можно добавить num_layers=2)
        self.lstm = nn.LSTM(embed_dim, hidden_size, batch_first=True)
        
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        
        # Распаковываем кортеж с двумя видами памяти
        out, (h_n, c_n) = self.lstm(emb)
        
        # Берем только краткосрочную память верхнего слоя
        final_memory = h_n[-1] 
        
        logits = self.fc(final_memory)
        return logits